# Capability 12: Hybrid data retrieval

6/6 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 12`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap12_hybrid_retrieval.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: numbers (structured) + qualitative commentary (unstructured) in one question

**✅ PASS**

In [ ]:
_ = ask('How did North America perform in Q1 2024, and what does the earnings commentary say about it?', '01')

01
USER: How did North America perform in Q1 2024, and what does the earnings commentary say about it?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-001', 'DOC-005', 'DOC-011', 'DOC-013', 'DOC-015']]

AGENT: ## North America — Q1 2024

**Headline metrics** (structured KPI data + commentary in [DOC-001]):

| Metric | Q1 2024 |
|---|---|
| Revenue | $3,593M |
| Organic revenue growth | **-8.8%** |
| Volume | 21,353K hL (~21.4M hL) |
| Normalized EBITDA | $1,126M |
| EBITDA margin | 31.3% (computed) |

Source: *AB InBev First Quarter 2024 Results* (BusinessWire, 7 May 2024).

**What the earnings commentary says**

The retrieved Q1 2024 zone-by-zone commentary [DOC-001] reports North America exactly in line with the structured data: revenue of $3,593M with **organic growth of -8.8%**, volume of 21,353K hL, and normalized EBITDA of $1,126M at a **31.3% margin**.

That is the full extent of the North America narrative in the retrieved material. Two import

## 02: structured KPI + a named real competitor (needs web)

**✅ PASS**

In [ ]:
_ = ask("What was AB InBev's revenue in 2025, and how does that compare to Heineken's public position?", '02')

02
USER: What was AB InBev's revenue in 2025, and how does that compare to Heineken's public position?

[intent=comparison | sub_agents=['structured', 'web']]
[assumptions/limitations surfaced:]
  - 'Heineken' isn't part of Anheuser-Busch InBev (AB InBev)'s tracked entities (brand/zone/country), or is a different company entirely (e.g. a competitor), so no internal structured data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting.

AGENT: ## AB InBev — FY2025 revenue (internal reporting)

| Metric | Value |
|---|---|
| Zone | Global |
| Period | FY2025 (annual) |
| Revenue | **USD 59,320.0M** |

Source: internal structured KPI data (query on `fact_kpi`, zone = Global, year = 2025, grain = annual).

## Comparison with Heineken — not supported by the available evidence

I can't give you a like-for-like comparison, and I'd rather flag that than approximate one:

- **No internal data exists for Heineken.** Heineken is not one of AB

## 03: structured + coding: a number plus a derived calculation on it

**✅ PASS**

In [ ]:
_ = ask("What was North America's Q1 2024 revenue, and what would it be if it grew 5% annually for 3 years?", '03')

03
USER: What was North America's Q1 2024 revenue, and what would it be if it grew 5% annually for 3 years?

[intent=data_query | sub_agents=['structured', 'coding']]

AGENT: ## North America — Q1 2024 Revenue

| Metric | Value |
| --- | --- |
| Q1 2024 revenue (North America) | **$3,593M** |
| Projected revenue after 3 years at 5% annual growth | **$4,159.35M** |

**Explanation:** Starting from the Q1 2024 revenue of $3,593M, applying 5% annual growth compounded over 3 years yields **$4,159.35M** (calculated as $3,593M × 1.05³). This represents an increase of roughly $566M over the baseline.

**Assumptions / caveats:**
- This is a simple compounding exercise applied to a single quarter's revenue — it is not a company forecast or guidance.
- It assumes the 5% growth rate applies uniformly each year to the same Q1 baseline, with no seasonality, FX, volume/price-mix effects, or structural changes factored in.
- The query was scoped to `zone = North America`, `year = 2024`, `quarter = 1`;

## 04: structured + unstructured + a hierarchy-fallback note (country -> zone)

**✅ PASS**

In [ ]:
_ = ask("What was AB InBev's revenue in Brazil specifically in 2025, and what's the commentary on it?", '04')

04
USER: What was AB InBev's revenue in Brazil specifically in 2025, and what's the commentary on it?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-011', 'DOC-013', 'DOC-005', 'DOC-004', 'DOC-012']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **South America** zone (which includes Brazil) instead, plus any Brazil-specific commentary found in the documents.

AGENT: ## Brazil revenue in 2025

**AB InBev does not publish structured revenue at the country level**, so there is no "Brazil-only" revenue line in the data. The closest available figure is the **South America zone** (of which Brazil is the largest market):

| Zone | Country (as tagged) | Year | Revenue (USD m) |
| --- | --- | --- | --- |
| South America | Brazil | 2025 | **$11,954** |

This $11,954M is a **zone-level total** (sum of the four quarterly results published in 2025 — Q1, Q2, Q3 and the Full Year/Q4 release

## 05: three sub-agents in one turn: structured + unstructured + web

**✅ PASS**

In [ ]:
_ = ask("How did North America perform in Q1 2024, what's Heineken's public market position, and any recent AB InBev press commentary on North America?", '05')

05
USER: How did North America perform in Q1 2024, what's Heineken's public market position, and any recent AB InBev press commentary on North America?

[intent=data_query | sub_agents=['structured', 'unstructured', 'web']]
[citations: ['DOC-014', 'DOC-001', 'DOC-003', 'DOC-005', 'DOC-008']]
[assumptions/limitations surfaced:]
  - 'Heineken' isn't part of Anheuser-Busch InBev (AB InBev)'s tracked entities (brand/zone/country), or is a different company entirely (e.g. a competitor), so no internal structured data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting.

AGENT: ## North America Performance — Q1 2024

Per AB InBev's First Quarter 2024 Results (BusinessWire, 7 May 2024), North America delivered:

| Metric | Q1 2024 |
| --- | --- |
| Revenue | $3,593M |
| Volume | 21,353K hL |
| Normalized EBITDA | $1,126M |
| EBITDA Margin | 31.3% |
| Organic Revenue Growth | **-8.8%** |

Q1 2024 was a soft quarter for the zone — revenue

## 06: brand (unstructured) + a competitor comparison (web) together

**✅ PASS**

In [ ]:
_ = ask('How does Corona compare to Heineken in terms of brand positioning?', '06')

06
USER: How does Corona compare to Heineken in terms of brand positioning?

[intent=comparison | sub_agents=['unstructured', 'web']]
[citations: ['DOC-014', 'DOC-012', 'DOC-008', 'DOC-004', 'DOC-011']]
[assumptions/limitations surfaced:]
  - 'Heineken' isn't part of Anheuser-Busch InBev (AB InBev)'s tracked entities (brand/zone/country), or is a different company entirely (e.g. a competitor), so no internal structured data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting.

AGENT: ## Short answer

The retrieved evidence does **not** contain a genuine brand-positioning comparison between Corona and Heineken. Here's what the evidence actually supports, and where it runs out.

## What the evidence does say

**Corona (internal coverage — partial)**
- Corona is one of AB InBev's four "megabrands" — the global/multi-country portfolio anchored by Budweiser, Corona, Stella Artois and Michelob Ultra [DOC-012].
- That megabrand portfoli